# Importation librairie 

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB, CategoricalNB
from sklearn.linear_model import BayesianRidge, ARDRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import math
import numpy as np

# Méthode Bayésienne de classification

## GaussianNB : pour features continues ~ loi normale.

In [ ]:
def naive_bayes_gaussian(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred

## MultinomialNB : pour variables discrètes (textes, comptages).

In [ ]:
def naive_bayes_multinomial(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = MultinomialNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred

## BernoulliNB : pour variables binaires.

In [ ]:
def naive_bayes_bernoulli(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = BernoulliNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred


## ComplementNB : variante robuste aux classes déséquilibrées.

In [ ]:
def naive_bayes_complement(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = ComplementNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred

## CategoricalNB : pour variables catégorielles finies.

In [ ]:
def naive_bayes_categorical(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = CategoricalNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return model, acc, y_pred

# Méthode de régression bayésienne

## BayesianRidge : régression linéaire avec a priori gaussiens sur les poids.

In [ ]:
def bayesian_ridge(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = BayesianRidge()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

## ARDRegression (Automatic Relevance Determination) : ajoute un mécanisme de sparsité bayésienne pour sélectionner les variables pertinentes.

In [ ]:
def ard_regression(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    model = ARDRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return model, mse, y_pred

# Méthode de catégorisation expérimental

In [ ]:
class BayesianClassifierCustom:
    def __init__(self, lr=0.01, n_iter=1000, custom_loss=None):
        """
        lr : taux d'apprentissage
        n_iter : nombre d'itérations
        custom_loss : fonction de coût personnalisée f(y, y_pred, w)
        """
        self.lr = lr
        self.n_iter = n_iter
        self.custom_loss = custom_loss
        self.w = None
        self.b = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n, p = X.shape
        self.w = np.zeros(p)
        self.b = 0

        for _ in range(self.n_iter):
            linear = X.dot(self.w) + self.b
            y_pred = self.sigmoid(linear)
            error = y_pred - y

            grad_w = (1/n) * X.T.dot(error)
            grad_b = (1/n) * np.sum(error)

            # update weights
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

    def predict_proba(self, X):
        return self.sigmoid(X.dot(self.w) + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def loss(self, X, y):
        y_pred = self.predict_proba(X)
        if self.custom_loss:
            return self.custom_loss(y, y_pred, self.w)
        # log-loss par défaut
        eps = 1e-8
        log_loss = -np.mean(y*np.log(y_pred+eps) + (1-y)*np.log(1-y_pred+eps))
        return log_loss


In [ ]:
np.random.seed(42)
X_cls = np.random.randn(100, 3)
y_cls = (X_cls[:,0] + X_cls[:,1] - X_cls[:,2] > 0).astype(int)

clf = BayesianClassifierCustom(lr=0.05, n_iter=1000)
clf.fit(X_cls, y_cls)
print("Accuracy :", accuracy_score(y_cls, clf.predict(X_cls)))

# Méthode de régression bayésienne expérimental

In [ ]:
class BayesianRegressionCustom:
    def __init__(self, lr=0.01, n_iter=1000, alpha=0.1, custom_loss=None):
        """
        lr : taux d'apprentissage
        n_iter : nombre d'itérations
        alpha : force du prior gaussien sur les poids
        custom_loss : fonction de coût personnalisée f(y, y_pred, w)
        """
        self.lr = lr
        self.n_iter = n_iter
        self.alpha = alpha
        self.custom_loss = custom_loss
        self.w = None
        self.b = None

    def fit(self, X, y):
        n, p = X.shape
        self.w = np.zeros(p)
        self.b = 0

        for _ in range(self.n_iter):
            y_pred = X.dot(self.w) + self.b
            error = y_pred - y

            grad_w = (1/n) * X.T.dot(error) + self.alpha * self.w  # prior gaussien
            grad_b = (1/n) * np.sum(error)

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

    def predict(self, X):
        return X.dot(self.w) + self.b

    def loss(self, X, y):
        y_pred = self.predict(X)
        if self.custom_loss:
            return self.custom_loss(y, y_pred, self.w)
        # loss par défaut = MSE + prior
        mse = np.mean((y - y_pred) ** 2)
        penalty = 0.5 * self.alpha * np.sum(self.w**2)
        return mse + penalty

In [ ]:
X_reg = np.random.randn(100, 3)
true_w = np.array([1.5, -2.0, 0.5])
y_reg = X_reg.dot(true_w) + np.random.randn(100) * 0.5

reg = BayesianRegressionCustom(lr=0.05, n_iter=1000, alpha=0.1)
reg.fit(X_reg, y_reg)
print("MSE :", mean_squared_error(y_reg, reg.predict(X_reg)))
